# Operator Spaces Demo

本 notebook 演示当前 `pyope` 中与局域算符空间相关的三个新能力：

1. `NO` 的多参数 / list 输入
2. `LocalOperatorBasis` 的固定权重基枚举与坐标提取
3. `DescendantSpace` 的固定权重后代空间生成
4. `RealizedGenerator` 与 free-field realization 下的独立性筛选


In [1]:
import sys
sys.path.insert(0, '../src')

import sympy as sp

from pyope import *

In [2]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## 1. 定义生成元

这里先用 Virasoro 生成元 `T` 和一个权重 1 的电流 `J` 做最小示例。


In [3]:
T = BasicOperator('T', conformal_weight=2)
J = BasicOperator('J', conformal_weight=1)
c = sp.symbols('c')

OPE[J, J] = MakeOPE([One, 0])
OPE[T, J] = MakeOPE([2 * J, d(J)])
OPE[T, T] = MakeOPE([c/2 * One, 0, 2 * T, d(T)])

In [4]:
OPE(T,T)

OPEData({4: One*c/2, 2: 2*T, 1: ∂T})

## 2. `NO` 现在支持多参数和列表输入

这使得右嵌套正规序的构造更直接。


In [6]:
expr_multi = NO(T, J, d(J), T)
expr_list = NO([T, J, d(J), T])
expr_named = NO_product(T, J, d(J), T)

print('NO(T, J, d(J), T)     =', expr_multi)
print('NO([T, J, d(J), T])   =', expr_list)
print('NO_product(...)       =', expr_named)
print('all equal?            =', expr_multi == expr_list == expr_named)


NO(T, J, d(J), T)     = NO(T,NO(J,NO(∂J,T)))
NO([T, J, d(J), T])   = NO(T,NO(J,NO(∂J,T)))
NO_product(...)       = NO(T,NO(J,NO(∂J,T)))
all equal?            = True


## 3. `LocalOperatorBasis`：固定权重局域算符基

我们构造由 `T` 和 `J` 生成的局域算符基，并查看固定权重的基元素。


In [21]:
basis_builder = LocalOperatorBasis([T, J], max_weight=6)

In [25]:
basis_builder.list(3)

[d(T), d^2(J), NO(J, NO(J,J)), NO(J, T), NO(∂J, J)]

## 4. 坐标提取

`coordinates(expr, weight)` 会把表达式投影到固定权重基上。


In [30]:
expr = NO(NO(J, J), J); expr
basis = basis_builder.list(3); basis

coords = basis_builder.coordinates(expr, weight=3); coords

NO(NO(J,J), J)

[d(T), d^2(J), NO(J, NO(J,J)), NO(J, T), NO(∂J, J)]

Matrix([
[0],
[1],
[1],
[0],
[0]])

## 5. `DescendantSpace`：固定权重后代空间

当前最小实现从 source 出发，迭代使用：

- 导数 $\partial$
- 与强生成元做正规序

然后在目标权重处去重并取线性无关张成集。


In [31]:
descendants = DescendantSpace(basis_builder)

generated_from_T = descendants.generate(T, 4); generated_from_T
basis_from_T = descendants.basis(T, 4); basis_from_T

[NO(J,∂T) + NO(∂J,T), d^2(T), NO(J, NO(J,T)), NO(J, ∂T), NO(T, T)]

[NO(J,∂T) + NO(∂J,T), d^2(T), NO(J, NO(J,T)), NO(J, ∂T), NO(T, T)]

## 6. 多 source 的 span

也可以把多个 source 一起送进 `span(...)`。


In [ ]:
span_from_J = descendants.span([J], 3)

print('span generated from J at weight 3:')
for op in span_from_J:
    print('  ', op)


## 7. `RealizedGenerator` 与 free-field realization

如果一个 VOA 的强生成元本身是自由场复合表达式，我们可以把它们声明成 `RealizedGenerator`。

然后：

- 先在抽象生成元语言里构造 basis
- 再把 basis 元素展开到 free fields
- 最后按 free-field basis 做线性独立性筛选

注意：basis 枚举会区分 bosonic / fermionic 原子。

- bosonic 原子可重复；若出现非正权 bosonic 原子，无约束 fixed-weight 枚举一般会失控
- fermionic 原子默认同一个原子至多取一次，因此像 `c, \partial c, \partial^2 c, ...` 这类 fermionic 原子可以安全参与

所以下面这个 `bc` 例子现在可以直接演示 realization 坐标。


In [3]:
clear_registry()
b = BasicOperator('b', fermionic=True, conformal_weight=1)
c = BasicOperator('c', fermionic=True, conformal_weight=0)
OPE[b,c] = MakeOPE([One])

T = RealizedGenerator(
    'T',
    realization= - NO(b, d(c)),
    conformal_weight=2,
)
J = RealizedGenerator(
    "J",
    realization=NO(b, c),
    conformal_weight=1,
)

abstract_basis = LocalOperatorBasis([T, J], max_weight=4)
free_field_basis = LocalOperatorBasis([b, c], max_weight=4)

free_field_basis.coordinates(simplify(realize(d(NO(T, b)))), weight=4)
simplify(realize(d(NO(T, b))))

Matrix([
[1/2],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0],
[  0]])

∂^3b/2

In [50]:
abstract_basis.list(4)

[d^2(T),
 d^3(J),
 NO(J, NO(J,NO(J,J))),
 NO(J, NO(J,T)),
 NO(J, ∂T),
 NO(T, T),
 NO(∂J, NO(J,J)),
 NO(∂J, T),
 NO(∂J, ∂J),
 NO(∂^2J, J)]

In [21]:
free_field_basis.list(1)

[BasicOperator('b', fermionic=True), d(c), NO(b, c), NO(∂c, c)]

In [24]:
abstract_basis_w4 = abstract_basis.list(4)
abstract_basis_w4


[d^2(T), NO(T, T)]

In [27]:
free_field_basis.coordinates(realize(abstract_basis_w4[0]), weight=4)


Matrix([
[ 0],
[ 0],
[ 0],
[ 0],
[ 0],
[-2],
[ 0],
[ 0],
[ 0],
[-4],
[-1],
[ 0],
[ 0],
[ 0],
[ 0],
[ 0],
[ 0],
[-5]])

In [52]:
independent_realized_basis = independent_under_realization(
    abstract_basis.list(2),
    free_field_basis=free_field_basis,
    weight=2,
)

print('independent basis under free-field realization:')
for op in independent_realized_basis:
    print('  ', op)

independent basis under free-field realization:
   ∂J
   NO(J,J)


In [18]:
T.realize() + (1/2) * simplify(d(J).realize()) - 0.5 * simplify(NO(J, J).realize())

0